In [19]:
#nm20260220: ILAT = apex-lat to calculate GCLAT GCLON
# Richmond, A. D. (1995), Ionospheric Electrodynamics Using Magnetic Apex Coordinates, Journal of geomagnetism and geoelectricity, 47(2), 191–212, doi:10.5636/jgg.47.191.
# https://apexpy.readthedocs.io/en/latest/readme.html
#
#FOOTPRINT_ALT_KM = 300.0 #altitude of the magnetic footprint GCLAT,GCLON
#Input Parameters: XXLAT, XXLON, ALT_km
#Calculated Parameters:
#GCLAT/GCLON
#ILAT
#GLAT
#GMLT

In [20]:
import numpy as np
import pandas as pd
from apexpy import Apex


def compute_apex_footprint_products(
    df: pd.DataFrame,
    xlat_fixed: float,
    xlon_fixed: float,
    footprint_alt_km: float = 300.0,  #this must match the Akebono orbit data
    refh_km: float = 110.0,  #must be fixed
    lon_convention: str = "east_0_360",   # "east_0_360" or "pm180"
    include_qd_glat: bool = True,
    verify_mapping: bool = True,
    verify_tol_deg: float = 1e-3,
) -> pd.DataFrame:
    """
    Production-grade Apex mapping for MI-coupling / auroral precipitation workflows.

    Fixed inputs:
      - xlat_fixed, xlon_fixed: geodetic spacecraft lat/lon (constant for all rows)
      - df['Altitude']: spacecraft altitude [km]
      - df['DateTimeFormatted']: timestamps (timezone-aware OK; interpreted as UTC)

    Outputs per row:
      - XXLAT, XXLON: fixed geodetic inputs
      - ILAT: invariant latitude = modified apex latitude (alat) at spacecraft point
      - ALON: modified apex longitude (alon) at spacecraft point (field-line label)
      - MLT: magnetic local time [h] computed from ALON and UTC time (Apex definition)
      - GCLAT, GCLON: geodetic footprint at `footprint_alt_km` on the same apex field line
                      (computed by converting (ILAT, ALON) apex->geo at that altitude)
      - GLAT (optional): QD latitude at spacecraft point (diagnostic "geomagnetic lat")
      - VERIFY_OK (optional): whether apex->geo footprint maps back to same (ILAT, ALON)

    Notes:
      - This uses an APEX-consistent footprint (hold (alat, alon) constant), which is
        appropriate when field-line mapping fidelity is critical.
      - `refh_km` defines the coordinate system (typically 110 km). Do not confuse it
        with `footprint_alt_km`.
    """
    # --------------------------
    # Input validation
    # --------------------------
    required = {"DateTimeFormatted", "Altitude"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"df is missing required columns: {sorted(missing)}")

    if lon_convention not in ("east_0_360", "pm180"):
        raise ValueError("lon_convention must be 'east_0_360' or 'pm180'")

    # Work on a copy to avoid mutating caller's DataFrame
    df = df.copy()

    # Ensure datetime column is datetime64 and timezone-naive UTC
    df["DateTimeFormatted"] = pd.to_datetime(df["DateTimeFormatted"], utc=True, errors="raise")
    # Remove tzinfo (ApexPy expects naive datetimes; still UTC by construction above)
    df["DateTimeFormatted"] = df["DateTimeFormatted"].dt.tz_convert(None)

    # Ensure Altitude is numeric
    alt = pd.to_numeric(df["Altitude"], errors="raise").to_numpy(dtype=float)

    # Normalize fixed longitude to [0, 360) for internal consistency
    xlat_fixed = float(xlat_fixed)
    xlon_fixed = float(xlon_fixed)
    xlon_fixed_360 = xlon_fixed % 360.0

    def _wrap360(x):
        return np.mod(x, 360.0)

    def _wrap180(x):
        return (x + 180.0) % 360.0 - 180.0

    n = len(df)
    times_py = df["DateTimeFormatted"].dt.to_pydatetime()

    # Preallocate outputs
    ILAT = np.full(n, np.nan, dtype=float)
    ALON = np.full(n, np.nan, dtype=float)
    MLT  = np.full(n, np.nan, dtype=float)
    GCLAT = np.full(n, np.nan, dtype=float)
    GCLON = np.full(n, np.nan, dtype=float)
    GLAT = np.full(n, np.nan, dtype=float) if include_qd_glat else None
    VERIFY_OK = np.full(n, True, dtype=bool) if verify_mapping else None

    # --------------------------
    # Main loop: group by day (good speed / stable IGRF epoch usage)
    # --------------------------
    day_groups = df.groupby(df["DateTimeFormatted"].dt.floor("D")).groups
    for day, idx in day_groups.items():
        A = Apex(pd.Timestamp(day).to_pydatetime(), refh=refh_km)

        ii = np.array(list(idx), dtype=int)
        k = len(ii)

        lat_i = np.full(k, xlat_fixed, dtype=float)
        lon_i = np.full(k, xlon_fixed_360, dtype=float)
        alt_i = alt[ii]
        t_i = [times_py[j] for j in ii]

        # 1) Spacecraft apex coords (field-line labels)
        alat_i, alon_i = A.geo2apex(lat_i, lon_i, alt_i)
        alat_i = np.asarray(alat_i, dtype=float)
        alon_i = _wrap360(np.asarray(alon_i, dtype=float))

        ILAT[ii] = alat_i
        ALON[ii] = alon_i

        # 2) MLT from apex longitude + time
        # ApexPy's mlon2mlt expects "magnetic longitude" + datetime
        mlt_i = np.array([A.mlon2mlt(a, tt) for a, tt in zip(alon_i, t_i)], dtype=float)
        MLT[ii] = np.mod(mlt_i, 24.0)

        # 3) Footprint at chosen altitude: (alat, alon) in apex -> GEO at height
        fp_lat, fp_lon = A.convert(alat_i, alon_i, "apex", "geo", height=footprint_alt_km)
        fp_lat = np.asarray(fp_lat, dtype=float)
        fp_lon = _wrap360(np.asarray(fp_lon, dtype=float))

        GCLAT[ii] = fp_lat
        GCLON[ii] = fp_lon

        # 4) Optional: diagnostic QD latitude at spacecraft
        if include_qd_glat:
            qdlat_i, _ = A.geo2qd(lat_i, lon_i, alt_i)
            GLAT[ii] = np.asarray(qdlat_i, dtype=float)

        # 5) Optional: verify mapping consistency (footprint maps back to same ILAT/ALON)
        if verify_mapping:
            alat_chk, alon_chk = A.geo2apex(fp_lat, fp_lon, np.full(k, footprint_alt_km, dtype=float))
            alat_chk = np.asarray(alat_chk, dtype=float)
            alon_chk = _wrap360(np.asarray(alon_chk, dtype=float))

            d_ilat = np.abs(alat_chk - alat_i)
            d_alon = np.abs(_wrap180(alon_chk - alon_i))

            VERIFY_OK[ii] = (d_ilat <= verify_tol_deg) & (d_alon <= verify_tol_deg)

    # --------------------------
    # Assemble output DataFrame
    # --------------------------
    out = pd.DataFrame(index=df.index)
    out["XXLAT"] = xlat_fixed
    out["XXLON"] = xlon_fixed if lon_convention == "pm180" else xlon_fixed_360

    out["ILAT"] = ILAT
    out["ALON"] = ALON
    out["MLT"] = MLT
    out["GCLAT"] = GCLAT
    out["GCLON"] = _wrap180(GCLON) if lon_convention == "pm180" else GCLON
    out["FOOTPRINT_ALT_KM"] = float(footprint_alt_km)

    if include_qd_glat:
        out["GLAT"] = GLAT
    if verify_mapping:
        out["VERIFY_OK"] = VERIFY_OK

    return out

In [21]:
#test1 version2: simplified output
import pandas as pd
import numpy as np

test_times = pd.date_range("1991-01-28 00:00:00", "1991-01-28 12:00:00", freq="4h", tz="UTC")
TEST_ALTITUDES = [1000.0, 4500.0, 8000.0]

df = pd.DataFrame({
    "DateTimeFormatted": np.repeat(test_times, len(TEST_ALTITUDES)),
    "Altitude": np.tile(TEST_ALTITUDES, len(test_times)),
})

out = compute_apex_footprint_products(
    df,
    xlat_fixed=43.0,
    xlon_fixed=-71.0,
    footprint_alt_km=300.0,
    refh_km=110.0,
    lon_convention="pm180",
    include_qd_glat=True,
    verify_mapping=True,
)

# Remove internal columns Blake does not need
out = out.drop(columns=["ALON", "FOOTPRINT_ALT_KM"], errors="ignore")

final_view = pd.concat([df, out], axis=1)
print(final_view.to_string(index=False))

        DateTimeFormatted  Altitude  XXLAT  XXLON      ILAT       MLT     GCLAT      GCLON      GLAT  VERIFY_OK
1991-01-28 00:00:00+00:00    1000.0   43.0  -71.0 56.190762 19.155182 44.893536 -71.667358 53.600872       True
1991-01-28 00:00:00+00:00    4500.0   43.0  -71.0 62.595512 19.025771 51.468189 -73.529884 53.408272       True
1991-01-28 00:00:00+00:00    8000.0   43.0  -71.0 66.400688 18.967414 55.440907 -74.427086 53.405972       True
1991-01-28 04:00:00+00:00    1000.0   43.0  -71.0 56.190762 23.329031 44.893536 -71.667358 53.600872       True
1991-01-28 04:00:00+00:00    4500.0   43.0  -71.0 62.595512 23.199620 51.468189 -73.529884 53.408272       True
1991-01-28 04:00:00+00:00    8000.0   43.0  -71.0 66.400688 23.141262 55.440907 -74.427086 53.405972       True
1991-01-28 08:00:00+00:00    1000.0   43.0  -71.0 56.190762  3.619254 44.893536 -71.667358 53.600872       True
1991-01-28 08:00:00+00:00    4500.0   43.0  -71.0 62.595512  3.489843 51.468189 -73.529884 53.408272    

/var/folders/7t/9j068l1n0vnfbltmw0nwvqmh004n7c/T/ipykernel_96570/1573956532.py:75: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  times_py = df["DateTimeFormatted"].dt.to_pydatetime()


In [22]:
# test2 with the real Akebono orbit data
import pandas as pd

# --- Akebono test inputs (single real point) ---
AKEBONO_XXLAT = 50.91
AKEBONO_XXLON = 10.2
AKEBONO_ALT_KM = 923.0
AKEBONO_TIME = pd.Timestamp("1991-01-28 02:01:37", tz="UTC")

df_akebono = pd.DataFrame({
    "DateTimeFormatted": [AKEBONO_TIME],
    "Altitude": [AKEBONO_ALT_KM],
})

out_akebono = compute_apex_footprint_products(
    df_akebono,
    xlat_fixed=AKEBONO_XXLAT,
    xlon_fixed=AKEBONO_XXLON,
    footprint_alt_km=300.0,
    refh_km=110.0,
    lon_convention="pm180",
    include_qd_glat=True,
    verify_mapping=True,
)

# Remove internal columns Blake does not need
out_akebono = out_akebono.drop(columns=["ALON", "FOOTPRINT_ALT_KM"], errors="ignore")

print("\n--- Akebono single-point test (real orbit point) ---")
print(pd.concat([df_akebono, out_akebono], axis=1).to_string(index=False))


--- Akebono single-point test (real orbit point) ---
        DateTimeFormatted  Altitude  XXLAT  XXLON      ILAT      MLT     GCLAT     GCLON      GLAT  VERIFY_OK
1991-01-28 02:01:37+00:00     923.0  50.91   10.2 49.939011 2.731645 53.144047 10.096198 46.938873       True


/var/folders/7t/9j068l1n0vnfbltmw0nwvqmh004n7c/T/ipykernel_96570/1573956532.py:75: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  times_py = df["DateTimeFormatted"].dt.to_pydatetime()


In [23]:
# modified column order to match the order in TEDformat.pdf
desired_order = [
    "DateTimeFormatted",
    "Altitude",
    "GCLAT",
    "GCLON",
    "ILAT",
    "GLAT",
    "MLT",
    "XXLAT",
    "XXLON",
]

final_view = pd.concat([df_akebono, out_akebono], axis=1)

# Keep only columns that exist, in the requested order
final_view = final_view[[c for c in desired_order if c in final_view.columns]]

print("\n--- Akebono output (reordered) ---")
print(final_view.to_string(index=False))


--- Akebono output (reordered) ---
        DateTimeFormatted  Altitude     GCLAT     GCLON      ILAT      GLAT      MLT  XXLAT  XXLON
1991-01-28 02:01:37+00:00     923.0 53.144047 10.096198 49.939011 46.938873 2.731645  50.91   10.2
